In [18]:
# random points generator
!python3 cli/data.py -m 700 -n 1 -se 1 -fi data/data_1D.csv
!python3 cli/data.py -m 700 -n 2 -se 50 -fi data/data_2D.csv
!python3 cli/data.py -m 700 -n 5 -se 1 -fi data/data_5D.csv

Successfully generated 700 points in 'data/data_1D.csv'
Successfully generated 700 points in 'data/data_2D.csv'
Successfully generated 700 points in 'data/data_5D.csv'


In [9]:
!python3 cli/visualise.py -da data/data_1D.csv -fi assets/data_1D.png
!python3 cli/visualise.py -da data/data_2D.csv -fi assets/data_2D.png
!python3 cli/visualise.py -da data/data_5D.csv -fi assets/data_5D.png

![](assets/data_5D.png "data 5D")   
*~ Pairplot of the dataset with 5 features. Generated using the CLI above, though I'm 99.9% sure there are a flurry of issues with it*

In [2]:
from pandas import DataFrame, read_csv
from sklearn.model_selection import train_test_split
import numpy as np

data: DataFrame = read_csv("data/data_5D.csv")
x: DataFrame = data[data.columns[1:]]
y: DataFrame = data[["y"]]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 45)
x_train: np.ndarray = np.array(x_train)
x_test: np.ndarray = np.array(x_test)
y_train: np.ndarray = np.array(y_train)
y_test: np.ndarray = np.array(y_test)

In [3]:
# Metrics declaration
import numpy as np

class Metrics():
  @classmethod
  def mean_squared(cls, error: np.ndarray) -> float:
    m: int = error.shape[0]
    return ((error.T @ error) * (1 / m)).item()

  @classmethod
  def root_mean_squared(cls, error: np.ndarray) -> float:
    return np.sqrt(cls.mean_squared(error))

  @classmethod
  def r_squared(cls, error: np.ndarray, variance: np.ndarray) -> float:
    return (1 - ((error.T @ error) / (variance.T @ variance))).item()

#### Multi-linear regression summary

Linear regression is based on the equation:

$$y = XM + C$$

where $M$ is of shape $(n, 1)$, $X$ is of shape $(m, n)$, $C$ is of shape $(1, 1)$ and thus $y$ is of shape $(m, 1)$ *(Numpy arrays are of shape (m, n) where m is the number of rows/samples and n is the number of columns/features)*. After the matrix multiplication, $XM$ is of shape $(m, 1)$. This gives the method to generate our $y$.

Gradient descent iteratively learns the *best* weights and biases. To know if its doing well in each iteration, it has to figure out the error between the true values, $y$ and its predicted values, $\hat y$ via the mean squared error:

$$ L = \frac{1}{m} (\hat y - y)^T (\hat y - y) = \frac{1}{m} (XM + C - y)^T (XM + C - y)$$

From the equation, we can differentiate with respect to M and C:

$$ \frac{\partial L}{\partial M} = \frac{2}{m} X^T(\hat{y} - y)$$

$$ \frac{\partial L}{\partial C} = \frac{2}{m} 1^T(\hat{y} - y)$$

where $1$ is of shape $(m, 1)$ So at each iteration we can update the weights and bias:

$$ M = M - lr \frac{\partial L}{\partial M}$$

$$ C = C - lr \frac{\partial L}{\partial C}$$

By applying this in each iteration, we are going against the gradient towards the minimum. The learning rate $lr$ defines the how quickly we apply these changes; this is a hyperparameter that we can tweak to our liking

In [ ]:
import numpy as np

class Multi_Linear_Regression:
  C: np.ndarray 
  M: np.ndarray 

  def train(self, x_train: np.ndarray, y_train: np.ndarray, lr: float = 1e-5, epoch: int = 1000) -> None:
    m, n = x_train.shape[0], x_train.shape[1]

    self.M: np.ndarray = np.ones((n, 1))
    self.C: np.ndarray = np.ones((1, 1))
    for _ in range(epoch): 
      y_hat: np.ndarray = (x_train @ self.M) + self.C
      error: np.ndarray = y_hat - y_train

      M_der: np.ndarray = (x_train.T @ error) * (2 / m)
      C_der: np.ndarray = (np.ones(m) @ error) * (2 / m)

      self.C: np.ndarray = self.C - (lr * C_der)
      self.M: np.ndarray = self.M - (lr * M_der)
  
  def predict(self, x_test: np.ndarray) -> np.ndarray:
    return (x_test @ self.M) + self.C

In [8]:
import numpy as np

multi_linear_regression: Multi_Linear_Regression = Multi_Linear_Regression()
multi_linear_regression.train(x_train, y_train)

mean: np.ndarray = np.mean(y_test, dtype = np.ndarray)
variance: np.ndarray = y_test - mean

y_hat: np.ndarray = multi_linear_regression.predict(x_test)
error: np.ndarray = y_hat - y_test

print(f"M: \n{multi_linear_regression.M}\n")
print(f"C: \n{multi_linear_regression.C}\n")

print(f"Mean squared error: {Metrics.mean_squared(error)}")
print(f"Root mean squared error: {Metrics.root_mean_squared(error)}")
print(f"R2 score: {Metrics.r_squared(error, variance)}")

M: 
[[-0.81062369]
 [ 2.20838306]
 [-5.06963665]
 [-1.92534619]
 [-3.45104472]]

C: 
[[1.10136752]]

Mean squared error: 3315.29626312869
Root mean squared error: 57.578609423367375
R2 score: 0.9780934911888347
